In [1]:
# CAPSTONE PROJECT 1:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Smart Retail Inventory and Sales Analytics") \
    .getOrCreate()

In [2]:
%%writefile stores.csv
store_id,store_name,city,state,store_type,manager_name
S101,Metro Mart Hyderabad,Hyderabad,Telangana,Supermarket,Rahul Sharma
S102,Metro Mart Bangalore,Bangalore,Karnataka,Supermarket,Priya Reddy
S103,Metro Mart Mumbai,Mumbai,Maharashtra,Hypermarket,Amit Kumar
S104,Metro Mart Chennai,Chennai,Tamil Nadu,Supermarket,Sneha Patel
S105,Metro Mart Delhi,Delhi,Delhi,Hypermarket,Farhan Ali
S106,Metro Mart Pune,Pune,Maharashtra,Mini Store,Neha Singh
S107,Metro Mart Kochi,Kochi,Kerala,Mini Store,Arjun Verma
S108,Metro Mart Jaipur,Jaipur,Rajasthan,Supermarket,Meera Nair

Writing stores.csv


In [3]:
%%writefile products.csv
product_id,product_name,category,brand,supplier_id,unit_price
P101,Laptop,Electronics,Lenovo,S201,65000
P102,Mobile,Electronics,Samsung,S202,25000
P103,Television,Electronics,LG,S203,45000
P104,Office Chair,Furniture,Featherlite,S204,7000
P105,Study Table,Furniture,Urban Ladder,S204,12000
P106,Shoes,Fashion,Nike,S205,4500
P107,Watch,Fashion,Fastrack,S206,8000
P108,Backpack,Fashion,Wildcraft,S206,2500
P109,Refrigerator,Electronics,Whirlpool,S203,38000
P110,Sofa,Furniture,Godrej,S204,32000
P111,Headphones,Electronics,Sony,S999,3000
P112,T-Shirt,Fashion,Puma,,1500

Writing products.csv


In [4]:
%%writefile inventory.csv
inventory_id,store_id,product_id,stock_quantity,reorder_level,last_update
I1001,S101,P101,10,5,2026-01-10
I1002,S101,P102,25,10,2026-01-10
I1003,S101,P104,3,5,2026-01-11
I1004,S102,P101,8,5,2026-01-12
I1005,S102,P103,5,4,2026-01-12
I1006,S103,P105,2,5,2026-01-13
I1007,S103,P106,30,10,2026-01-14
I1008,S104,P107,4,5,2026-01-15
I1009,S105,P108,50,20,2026-01-15
I1010,S106,P109,,6,2026-01-16
I1011,S107,P110,1,3,2026-01-17
I1012,S108,P120,12,5,2026-01-18

Writing inventory.csv


In [5]:
%%writefile sales.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_m
SA1001,S101,P101,2026-01-10,1,65000,UPI
SA1002,S101,P102,2026-01-10,2,50000,Card
SA1003,S102,P101,2026-01-11,1,65000,UPI
SA1004,S103,P106,2026-01-12,4,18000,Cash
SA1005,S104,P107,2026-01-12,1,8000,Card
SA1006,S105,P108,2026-01-13,5,12500,UPI
SA1007,S106,P109,2026-01-14,1,38000,Card
SA1008,S107,P110,2026-01-15,1,32000,UPI
SA1009,S108,P120,2026-01-15,2,10000,Cash
SA1010,S101,P104,2026-01-16,2,14000,
SA1011,S102,P103,2026-01-17,1,,UPI
SA1012,S103,P105,2026-01-18,1,12000,Card
SA1013,S104,P107,2026-02-01,2,16000,UPI
SA1014,S105,P108,2026-02-02,3,7500,Cash
SA1015,S101,P102,2026-02-03,1,25000,Card

Writing sales.csv


In [6]:
%%writefile suppliers.json
[
{
"supplier_id":"S201",
"supplier_name":"TechSource India",
"city":"Hyderabad",
"rating":4.5,
"contact":{"phone":"9876500011","email":"techsource@mail.com"}
},
{
"supplier_id":"S202",
"supplier_name":"MobileWorld Distributors",
"city":"Bangalore",
"rating":4.2,
"contact":{"phone":null,"email":"mobileworld@mail.com"}
},
{
"supplier_id":"S203",
"supplier_name":"HomeTech Supply",
"city":"Mumbai",
"rating":4.4,
"contact":{"phone":"9876500013","email":null}
},
{
"supplier_id":"S204",
"supplier_name":"Urban Furniture Co",
"city":"Delhi",
"rating":4.0,
"contact":{"phone":"9876500014","email":"urban@mail.com"}
},
{
"supplier_id":"S205",
"supplier_name":"Fashion Direct",
"city":"Pune",
"rating":3.8,
"contact":{"phone":null,"email":null}
}
]

Writing suppliers.json


In [16]:
# PART 1: INGESTION
stores_df = spark.read.csv("stores.csv", header=True, inferSchema=True)
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)
inventory_df = spark.read.csv("inventory.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [9]:
suppliers_df = spark.read.option(
    "multiline",
    "true"
).json("suppliers.json")

In [10]:
stores_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)



In [12]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)



In [13]:
inventory_df.printSchema()

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)



In [14]:
sales_df.printSchema()

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- payment_m: string (nullable = true)



In [15]:
suppliers_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)



In [17]:
stores_df.count()

8

In [18]:
products_df.count()

12

In [19]:
inventory_df.count()

12

In [20]:
sales_df.count()

15

In [21]:
suppliers_df.count()

5

In [22]:
stores_df.write.mode("overwrite").parquet("bronze/stores")

In [23]:
products_df.write.mode("overwrite").parquet("bronze/products")

In [24]:
inventory_df.write.mode("overwrite").parquet("bronze/inventory")

In [25]:
sales_df.write.mode("overwrite").parquet("bronze/sales")

In [26]:
suppliers_df.write.mode("overwrite").parquet("bronze/suppliers")

In [27]:
spark.read.parquet("bronze/stores").show()

spark.read.parquet("bronze/products").show()

spark.read.parquet("bronze/inventory").show()

spark.read.parquet("bronze/sales").show()

spark.read.parquet("bronze/suppliers").show()

+--------+--------------------+---------+-----------+-----------+------------+
|store_id|          store_name|     city|      state| store_type|manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya Reddy|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|Hypermarket|  Amit Kumar|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|Supermarket| Sneha Patel|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|Hypermarket|  Farhan Ali|
|    S106|     Metro Mart Pune|     Pune|Maharashtra| Mini Store|  Neha Singh|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala| Mini Store| Arjun Verma|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|Supermarket|  Meera Nair|
+--------+--------------------+---------+-----------+-----------+------------+

+----------+------------+-----------+------------+-

In [28]:
# PART 2: DATA CLEANING
from pyspark.sql.functions import col
products_df.filter(
    col("supplier_id").isNull()
).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [29]:
inventory_df.filter(
    col("stock_quantity").isNull()
).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [30]:
sales_df.filter(
    col("sale_amount").isNull()
).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|      UPI|
+-------+--------+----------+----------+-------------+-----------+---------+



In [32]:
sales_df.columns

['sale_id',
 'store_id',
 'product_id',
 'sale_date',
 'quantity_sold',
 'sale_amount',
 'payment_m']

In [33]:
sales_df.filter(
    col("payment_m").isNull()
).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|     NULL|
+-------+--------+----------+----------+-------------+-----------+---------+



In [34]:
inventory_clean_df = inventory_df.fillna(
    {"stock_quantity": 0}
)

In [35]:
sales_clean_df = sales_df.fillna(
    {"sale_amount": 0}
)

In [36]:
sales_clean_df = sales_clean_df.fillna(
    {"payment_m": "Not Provided"}
)

In [37]:
products_clean_df = products_df.fillna(
    {"supplier_id": "UNKNOWN"}
)

In [38]:
from pyspark.sql.functions import when, col
products_clean_df = products_clean_df.withColumn(
    "data_quality_status",
    when(col("supplier_id") == "UNKNOWN", "Issue")
    .otherwise("Valid")
)
products_clean_df.show()

+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P103|  Television|Electronics|          LG|       S203|     45000|              Valid|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|              Valid|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|              Valid|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|              Valid|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|              Valid|
|      P108|    Backpack|    Fashion|   Wildcraft|       S20

In [39]:
inventory_clean_df = inventory_clean_df.withColumn(
    "data_quality_status",
    when(col("stock_quantity") == 0, "Issue")
    .otherwise("Valid")
)

inventory_clean_df.show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|              Valid|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|              Valid|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|              Valid|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|              Valid|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|              Valid|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|              Valid|
|       I1007|    S103|      P106|            30|           10| 2026-01-14|              Valid|
|       I1008|    S104|      P107|      

In [40]:
sales_clean_df = sales_clean_df.withColumn(
    "data_quality_status",
    when(
        (col("sale_amount") == 0) |
        (col("payment_m") == "Not Provided"),
        "Issue"
    ).otherwise("Valid")
)

sales_clean_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|         UPI|              Valid|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|        Card|              Valid|


In [41]:
products_clean_df.write.mode("overwrite").parquet(
    "silver/products"
)

In [42]:
inventory_clean_df.write.mode("overwrite").parquet(
    "silver/inventory"
)

In [43]:
sales_clean_df.write.mode("overwrite").parquet(
    "silver/sales"
)

In [44]:
stores_df.write.mode("overwrite").parquet(
    "silver/stores"
)

In [45]:
suppliers_df.write.mode("overwrite").parquet(
    "silver/suppliers"
)

In [46]:
spark.read.parquet("silver/products").show()

spark.read.parquet("silver/inventory").show()

spark.read.parquet("silver/sales").show()

+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P103|  Television|Electronics|          LG|       S203|     45000|              Valid|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|              Valid|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|              Valid|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|              Valid|
|      P107|       Watch|    Fashion|    Fastrack|       S206|      8000|              Valid|
|      P108|    Backpack|    Fashion|   Wildcraft|       S20

In [47]:
# PART 3: JSON FLATTENING
suppliers_flat_df = suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    "contact.phone",
    "contact.email"
)
suppliers_flat_df.show()

+-----------+--------------------+---------+------+----------+--------------------+
|supplier_id|       supplier_name|     city|rating|     phone|               email|
+-----------+--------------------+---------+------+----------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|      NULL|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|9876500013|                NULL|
|       S204|  Urban Furniture Co|    Delhi|   4.0|9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|      NULL|                NULL|
+-----------+--------------------+---------+------+----------+--------------------+



In [48]:
suppliers_flat_df.select(
    "supplier_id",
    "supplier_name",
    "phone"
).show()

+-----------+--------------------+----------+
|supplier_id|       supplier_name|     phone|
+-----------+--------------------+----------+
|       S201|    TechSource India|9876500011|
|       S202|MobileWorld Distr...|      NULL|
|       S203|     HomeTech Supply|9876500013|
|       S204|  Urban Furniture Co|9876500014|
|       S205|      Fashion Direct|      NULL|
+-----------+--------------------+----------+



In [49]:
suppliers_flat_df.select(
    "supplier_id",
    "supplier_name",
    "email"
).show()

+-----------+--------------------+--------------------+
|supplier_id|       supplier_name|               email|
+-----------+--------------------+--------------------+
|       S201|    TechSource India| techsource@mail.com|
|       S202|MobileWorld Distr...|mobileworld@mail.com|
|       S203|     HomeTech Supply|                NULL|
|       S204|  Urban Furniture Co|      urban@mail.com|
|       S205|      Fashion Direct|                NULL|
+-----------+--------------------+--------------------+



In [50]:
suppliers_flat_df.select(
    "supplier_id",
    "supplier_name",
    "email"
).show()

+-----------+--------------------+--------------------+
|supplier_id|       supplier_name|               email|
+-----------+--------------------+--------------------+
|       S201|    TechSource India| techsource@mail.com|
|       S202|MobileWorld Distr...|mobileworld@mail.com|
|       S203|     HomeTech Supply|                NULL|
|       S204|  Urban Furniture Co|      urban@mail.com|
|       S205|      Fashion Direct|                NULL|
+-----------+--------------------+--------------------+



In [51]:
suppliers_flat_df = suppliers_flat_df.fillna(
    {"phone": "Not Provided"}
)
suppliers_flat_df.show()

+-----------+--------------------+---------+------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|       phone|               email|
+-----------+--------------------+---------+------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|  9876500013|                NULL|
|       S204|  Urban Furniture Co|    Delhi|   4.0|  9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|Not Provided|                NULL|
+-----------+--------------------+---------+------+------------+--------------------+



In [52]:
suppliers_flat_df = suppliers_flat_df.fillna(
    {"email": "Not Provided"}
)
suppliers_flat_df.show()

+-----------+--------------------+---------+------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|       phone|               email|
+-----------+--------------------+---------+------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|  9876500013|        Not Provided|
|       S204|  Urban Furniture Co|    Delhi|   4.0|  9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|Not Provided|        Not Provided|
+-----------+--------------------+---------+------+------------+--------------------+



In [53]:
suppliers_flat_df.write.mode("overwrite").parquet(
    "silver/suppliers_flat"
)

In [54]:
spark.read.parquet(
    "silver/suppliers_flat"
).show(truncate=False)

+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013  |Not Provided        |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Not Provided|Not Provided        |
+-----------+------------------------+---------+------+------------+--------------------+



In [55]:
# PART 4: JOINS
products_suppliers_df=products_clean_df.join(
    suppliers_flat_df,
    on="supplier_id",
    how="left"
)
products_suppliers_df.show()

+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|       supplier_name|     city|rating|       phone|               email|
+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|              Valid|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|              Valid|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|
|       S203|      P103|  Television|Electronics|          LG|     45000|              Valid|     HomeTech Supply|   Mumbai|   4.4|  9876500013|        Not Provided|
|   

In [56]:
inventory_products_df=inventory_clean_df.join(
    products_clean_df,
    on="product_id",
    how="left"
)
inventory_products_df.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P104|       I1003|    S101|             3|            5| 2026-01-11|              Valid|Office Chair|  Furni

In [57]:
sales_stores_df=sales_clean_df.join(
    stores_df,
    on="store_id",
    how="left"
)
sales_stores_df.show()

+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|          store_name|     city|      state| store_type|manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|    S101| SA1001|      P101|2026-01-10|            1|      65000|         UPI|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S101| SA1002|      P102|2026-01-10|            2|      50000|        Card|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102| SA1003|      P101|2026-01-11|            1|      65000|         UPI|              Valid|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya

In [58]:
sales_products_df=sales_clean_df.join(
    products_clean_df,
    on="product_id",
    how="left"
)
sales_products_df.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P101| SA1003|    S102|2026-01-11|            1|      65000|         UPI|              Va

In [59]:
retail_sales_df=sales_clean_df\
.join(stores_df,on="store_id",how="left")\
.join(products_clean_df,on="product_id",how="left")\
.join(suppliers_flat_df,on="supplier_id",how="left")
retail_sales_df.show(truncate=False)

+-----------+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|supplier_id|product_id|store_id|sale_id|sale_date |quantity_sold|sale_amount|payment_m   |data_quality_status|store_name          |city     |state      |store_type |manager_name|product_name|category   |brand       |unit_price|data_quality_status|supplier_name           |city     |rating|phone       |email               |
+-----------+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+-------------------+------------------------+---------+------+------------+--------------------+
|S201       |P101      |S

In [60]:
products_clean_df.join(
    suppliers_flat_df,
    on="supplier_id",
    how="left_anti"
).show()

+-----------+----------+------------+-----------+---------+----------+-------------------+
|supplier_id|product_id|product_name|   category|    brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+---------+----------+-------------------+
|       S206|      P107|       Watch|    Fashion| Fastrack|      8000|              Valid|
|       S206|      P108|    Backpack|    Fashion|Wildcraft|      2500|              Valid|
|       S999|      P111|  Headphones|Electronics|     Sony|      3000|              Valid|
|    UNKNOWN|      P112|     T-Shirt|    Fashion|     Puma|      1500|              Issue|
+-----------+----------+------------+-----------+---------+----------+-------------------+



In [61]:
inventory_clean_df.join(
    products_clean_df,
    on="product_id",
    how="left_anti"
).show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|              Valid|
+----------+------------+--------+--------------+-------------+-----------+-------------------+



In [62]:
sales_clean_df.join(
    products_clean_df,
    on="product_id",
    how="left_anti"
).show()

+----------+-------+--------+----------+-------------+-----------+---------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+---------+-------------------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|     Cash|              Valid|
+----------+-------+--------+----------+-------------+-----------+---------+-------------------+



In [63]:
sales_clean_df.join(
    stores_df,
    on="store_id",
    how="left_anti"
).show()

+--------+-------+----------+---------+-------------+-----------+---------+-------------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|
+--------+-------+----------+---------+-------------+-----------+---------+-------------------+
+--------+-------+----------+---------+-------------+-----------+---------+-------------------+



In [64]:
# PART 5: TRANSFORMATIONS
inventory_products_df=inventory_products_df.withColumn(
    "stock_status",
    when(col("stock_quantity")<=col("reorder_level"),"Reorder Required")
    .otherwise("Sufficient Stock")
)
inventory_products_df.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|    stock_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|Sufficient Stock|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|Sufficient Stock|
|      P104|       I1003|    S

In [65]:
products_suppliers_df=products_suppliers_df.withColumn(
    "price_category",
    when(col("unit_price")>=50000,"Premium")
    .when(col("unit_price")>=10000,"Standard")
    .otherwise("Budget")
)

products_suppliers_df.show()

+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+--------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|       supplier_name|     city|rating|       phone|               email|price_category|
+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+--------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|              Valid|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|       Premium|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|              Valid|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|      Standard|
|       S203|      P103|  Television|Electronics|          LG|     45000|              Valid|  

In [66]:
sales_products_df=sales_products_df.withColumn(
    "revenue_category",
    when(col("sale_amount")>=50000,"High Revenue")
    .when(col("sale_amount")>=15000,"Medium Revenue")
    .otherwise("Low Revenue")
)

sales_products_df.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|revenue_category|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|    High Revenue|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|    High Revenue|
|      P10

In [67]:
from pyspark.sql.functions import month
sales_products_df=sales_products_df.withColumn(
    "month",
    month("sale_date"))
sales_products_df.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+-----+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|revenue_category|month|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+-----+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|    High Revenue|    1|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|    

In [68]:
from pyspark.sql.functions import year
sales_products_df=sales_products_df.withColumn(
    "year",
    year("sale_date"))
sales_products_df.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+-----+----+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|revenue_category|month|year|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+-----+----+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|    High Revenue|    1|2026|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|    

In [69]:
inventory_products_df=inventory_products_df.withColumn(
    "inventory_value",
    col("stock_quantity")*col("unit_price"))
inventory_products_df.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+---------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|    stock_status|inventory_value|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+---------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|Sufficient Stock|         650000|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|    

In [70]:
products_suppliers_df=products_suppliers_df.withColumn(
    "supplier_quality",
    when(col("rating")>=4.5,"Excellent")
    .when(col("rating")>=4.0,"Good")
    .otherwise("Average"))
products_suppliers_df.show()

+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+--------------+----------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|       supplier_name|     city|rating|       phone|               email|price_category|supplier_quality|
+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+--------------+----------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|              Valid|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|       Premium|       Excellent|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|              Valid|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|      Standard|            Good|
|       S2

In [71]:
# PART 6: AGGREGATIONS
stores_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
|Maharashtra|    2|
+-----------+-----+



In [72]:
products_clean_df.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
|  Furniture|    3|
+-----------+-----+



In [73]:
products_clean_df.groupBy("brand").count().show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [74]:
inventory_products_df.groupBy("store_id").sum("inventory_value").show()

+--------+--------------------+
|store_id|sum(inventory_value)|
+--------+--------------------+
|    S105|              125000|
|    S102|              745000|
|    S106|                   0|
|    S104|               32000|
|    S107|               32000|
|    S101|             1296000|
|    S108|                NULL|
|    S103|              159000|
+--------+--------------------+



In [75]:
inventory_products_df.groupBy("category").sum("inventory_value").show()

+-----------+--------------------+
|   category|sum(inventory_value)|
+-----------+--------------------+
|    Fashion|              292000|
|       NULL|                NULL|
|Electronics|             2020000|
|  Furniture|               77000|
+-----------+--------------------+



In [76]:
inventory_products_df.filter(
    col("stock_status")=="Reorder Required"
).count()

5

In [77]:
sales_clean_df.agg(
    {"sale_amount":"sum"}
).show()

+----------------+
|sum(sale_amount)|
+----------------+
|          373000|
+----------------+



In [78]:
sales_stores_df.groupBy("store_name").sum("sale_amount").show()

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|Metro Mart Bangalore|           65000|
|    Metro Mart Kochi|           32000|
|   Metro Mart Jaipur|           10000|
|   Metro Mart Mumbai|           30000|
|     Metro Mart Pune|           38000|
|Metro Mart Hyderabad|          154000|
|    Metro Mart Delhi|           20000|
|  Metro Mart Chennai|           24000|
+--------------------+----------------+



In [79]:
sales_stores_df.groupBy("city").sum("sale_amount").show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Bangalore|           65000|
|    Kochi|           32000|
|  Chennai|           24000|
|   Mumbai|           30000|
|     Pune|           38000|
|    Delhi|           20000|
|Hyderabad|          154000|
|   Jaipur|           10000|
+---------+----------------+



In [80]:
sales_products_df.groupBy("category").sum("sale_amount").show()

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|    Fashion|           62000|
|       NULL|           10000|
|Electronics|          243000|
|  Furniture|           58000|
+-----------+----------------+



In [81]:
sales_products_df.groupBy("product_name").sum("sale_amount").show()

+------------+----------------+
|product_name|sum(sale_amount)|
+------------+----------------+
|Office Chair|           14000|
|        NULL|           10000|
|Refrigerator|           38000|
|      Laptop|          130000|
|        Sofa|           32000|
|    Backpack|           20000|
|       Shoes|           18000|
|      Mobile|           75000|
|  Television|               0|
| Study Table|           12000|
|       Watch|           24000|
+------------+----------------+



In [82]:
sales_clean_df.groupBy("payment_m").sum("sale_amount").show()

+------------+----------------+
|   payment_m|sum(sale_amount)|
+------------+----------------+
|        Card|          133000|
|        Cash|           35500|
|Not Provided|           14000|
|         UPI|          190500|
+------------+----------------+



In [83]:
sales_products_df.groupBy("product_name") \
.sum("sale_amount") \
.orderBy(col("sum(sale_amount)").desc()) \
.show(1)

+------------+----------------+
|product_name|sum(sale_amount)|
+------------+----------------+
|      Laptop|          130000|
+------------+----------------+
only showing top 1 row


In [84]:
sales_stores_df.groupBy("store_name") \
.sum("sale_amount") \
.orderBy(col("sum(sale_amount)").desc()) \
.show(1)

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|Metro Mart Hyderabad|          154000|
+--------------------+----------------+
only showing top 1 row


In [85]:
sales_products_df.groupBy("category") \
.sum("sale_amount") \
.orderBy(col("sum(sale_amount)").desc()) \
.show(1)

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|Electronics|          243000|
+-----------+----------------+
only showing top 1 row


In [86]:
# PART 7: WINDOW FUNCTIONS
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,sum

product_revenue_df=sales_products_df.groupBy("product_name").agg(sum("sale_amount").alias("revenue"))
window_spec=Window.orderBy(col("revenue").desc())
product_revenue_df=product_revenue_df.withColumn("rank",rank().over(window_spec))
product_revenue_df.show()

+------------+-------+----+
|product_name|revenue|rank|
+------------+-------+----+
|      Laptop| 130000|   1|
|      Mobile|  75000|   2|
|Refrigerator|  38000|   3|
|        Sofa|  32000|   4|
|       Watch|  24000|   5|
|    Backpack|  20000|   6|
|       Shoes|  18000|   7|
|Office Chair|  14000|   8|
| Study Table|  12000|   9|
|        NULL|  10000|  10|
|  Television|      0|  11|
+------------+-------+----+



In [87]:
store_revenue_df=sales_stores_df.groupBy("store_name").agg(sum("sale_amount").alias("revenue"))
window_spec=Window.orderBy(col("revenue").desc())
store_revenue_df=store_revenue_df.withColumn("rank",rank().over(window_spec))
store_revenue_df.show()

+--------------------+-------+----+
|          store_name|revenue|rank|
+--------------------+-------+----+
|Metro Mart Hyderabad| 154000|   1|
|Metro Mart Bangalore|  65000|   2|
|     Metro Mart Pune|  38000|   3|
|    Metro Mart Kochi|  32000|   4|
|   Metro Mart Mumbai|  30000|   5|
|  Metro Mart Chennai|  24000|   6|
|    Metro Mart Delhi|  20000|   7|
|   Metro Mart Jaipur|  10000|   8|
+--------------------+-------+----+



In [88]:
category_product_df=sales_products_df.groupBy("category","product_name").agg(sum("sale_amount").alias("revenue"))
window_spec=Window.partitionBy("category").orderBy(col("revenue").desc())
category_product_df=category_product_df.withColumn("rank",rank().over(window_spec))
category_product_df.show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|Electronics|      Mobile|  75000|   2|
|Electronics|Refrigerator|  38000|   3|
|Electronics|  Television|      0|   4|
|    Fashion|       Watch|  24000|   1|
|    Fashion|    Backpack|  20000|   2|
|    Fashion|       Shoes|  18000|   3|
|  Furniture|        Sofa|  32000|   1|
|  Furniture|Office Chair|  14000|   2|
|  Furniture| Study Table|  12000|   3|
+-----------+------------+-------+----+



In [89]:
from pyspark.sql.functions import row_number

category_product_df=sales_products_df.groupBy("category","product_name").agg(sum("sale_amount").alias("revenue"))
window_spec=Window.partitionBy("category").orderBy(col("revenue").desc())
top_product_df=category_product_df.withColumn("row_num",row_number().over(window_spec))
top_product_df.filter(col("row_num")==1).show()

+-----------+------------+-------+-------+
|   category|product_name|revenue|row_num|
+-----------+------------+-------+-------+
|       NULL|        NULL|  10000|      1|
|Electronics|      Laptop| 130000|      1|
|    Fashion|       Watch|  24000|      1|
|  Furniture|        Sofa|  32000|      1|
+-----------+------------+-------+-------+



In [90]:
category_product_df=sales_products_df.groupBy("category","product_name").agg(sum("sale_amount").alias("revenue"))
window_spec=Window.partitionBy("category").orderBy(col("revenue").desc())
top3_products_df=category_product_df.withColumn("row_num",row_number().over(window_spec))
top3_products_df.filter(col("row_num")<=3).show()

+-----------+------------+-------+-------+
|   category|product_name|revenue|row_num|
+-----------+------------+-------+-------+
|       NULL|        NULL|  10000|      1|
|Electronics|      Laptop| 130000|      1|
|Electronics|      Mobile|  75000|      2|
|Electronics|Refrigerator|  38000|      3|
|    Fashion|       Watch|  24000|      1|
|    Fashion|    Backpack|  20000|      2|
|    Fashion|       Shoes|  18000|      3|
|  Furniture|        Sofa|  32000|      1|
|  Furniture|Office Chair|  14000|      2|
|  Furniture| Study Table|  12000|      3|
+-----------+------------+-------+-------+



In [91]:
store_state_df=sales_stores_df.groupBy("state","store_name").agg(sum("sale_amount").alias("revenue"))
window_spec=Window.partitionBy("state").orderBy(col("revenue").desc())
top_store_df=store_state_df.withColumn("row_num",row_number().over(window_spec))
top_store_df.filter(col("row_num")==1).show()

+-----------+--------------------+-------+-------+
|      state|          store_name|revenue|row_num|
+-----------+--------------------+-------+-------+
|      Delhi|    Metro Mart Delhi|  20000|      1|
|  Karnataka|Metro Mart Bangalore|  65000|      1|
|     Kerala|    Metro Mart Kochi|  32000|      1|
|Maharashtra|     Metro Mart Pune|  38000|      1|
|  Rajasthan|   Metro Mart Jaipur|  10000|      1|
| Tamil Nadu|  Metro Mart Chennai|  24000|      1|
|  Telangana|Metro Mart Hyderabad| 154000|      1|
+-----------+--------------------+-------+-------+



In [92]:
daily_sales_df=sales_clean_df.groupBy("sale_date").agg(sum("sale_amount").alias("daily_revenue"))
window_spec=Window.orderBy("sale_date")
daily_sales_df=daily_sales_df.withColumn("running_total",sum("daily_revenue").over(window_spec))
daily_sales_df.show()

+----------+-------------+-------------+
| sale_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2026-01-10|       115000|       115000|
|2026-01-11|        65000|       180000|
|2026-01-12|        26000|       206000|
|2026-01-13|        12500|       218500|
|2026-01-14|        38000|       256500|
|2026-01-15|        42000|       298500|
|2026-01-16|        14000|       312500|
|2026-01-17|            0|       312500|
|2026-01-18|        12000|       324500|
|2026-02-01|        16000|       340500|
|2026-02-02|         7500|       348000|
|2026-02-03|        25000|       373000|
+----------+-------------+-------------+



In [93]:
from pyspark.sql.functions import lag

daily_sales_df=sales_clean_df.groupBy("sale_date").agg(sum("sale_amount").alias("daily_revenue"))
window_spec=Window.orderBy("sale_date")
daily_sales_df=daily_sales_df.withColumn("previous_day_sales",lag("daily_revenue",1).over(window_spec))
daily_sales_df.show()

+----------+-------------+------------------+
| sale_date|daily_revenue|previous_day_sales|
+----------+-------------+------------------+
|2026-01-10|       115000|              NULL|
|2026-01-11|        65000|            115000|
|2026-01-12|        26000|             65000|
|2026-01-13|        12500|             26000|
|2026-01-14|        38000|             12500|
|2026-01-15|        42000|             38000|
|2026-01-16|        14000|             42000|
|2026-01-17|            0|             14000|
|2026-01-18|        12000|                 0|
|2026-02-01|        16000|             12000|
|2026-02-02|         7500|             16000|
|2026-02-03|        25000|              7500|
+----------+-------------+------------------+



In [94]:
from pyspark.sql.functions import lead
window_spec=Window.orderBy("sale_date")
sales_clean_df=sales_clean_df.withColumn("next_sale_amount",lead("sale_amount",1).over(window_spec))
sales_clean_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|next_sale_amount|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|           50000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|           65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|           18000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|            8000|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|           12500|
| SA1006|    S105|      P108|2026-01-13|            5|      1250

In [95]:
product_sales_df=sales_products_df.select("product_name","sale_date","sale_amount")
window_spec=Window.partitionBy("product_name").orderBy("sale_date")
product_sales_df=product_sales_df.withColumn("previous_sale_amount",lag("sale_amount",1).over(window_spec))
product_sales_df.filter(col("sale_amount")>col("previous_sale_amount")).show()

+------------+----------+-----------+--------------------+
|product_name| sale_date|sale_amount|previous_sale_amount|
+------------+----------+-----------+--------------------+
|       Watch|2026-02-01|      16000|                8000|
+------------+----------+-----------+--------------------+



In [97]:
stores_df.createOrReplaceTempView("stores")
products_df.createOrReplaceTempView("products")
inventory_df.createOrReplaceTempView("inventory")
sales_df.createOrReplaceTempView("sales")
suppliers_df.createOrReplaceTempView("suppliers")

In [98]:
spark.sql("SELECT * FROM sales").show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|      UPI|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|     Card|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|      UPI|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|     Cash|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|     Card|
| SA1006|    S105|      P108|2026-01-13|            5|      12500|      UPI|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|     Card|
| SA1008|    S107|      P110|2026-01-15|            1|      32000|      UPI|
| SA1009|    S108|      P120|2026-01-15|            2|      10000|     Cash|
| SA1010|    S101|      P104|2026-01-16|            2|      14000|     NULL|

In [99]:
spark.sql("""
SELECT category, COUNT(*) AS product_count
FROM products
GROUP BY category
""").show()

+-----------+-------------+
|   category|product_count|
+-----------+-------------+
|    Fashion|            4|
|Electronics|            5|
|  Furniture|            3|
+-----------+-------------+



In [100]:
spark.sql("""
SELECT s.store_id, s.store_name,
       SUM(sa.sale_amount) AS revenue
FROM sales sa
JOIN stores s ON sa.store_id = s.store_id
GROUP BY s.store_id, s.store_name
ORDER BY revenue DESC
""").show()

+--------+--------------------+-------+
|store_id|          store_name|revenue|
+--------+--------------------+-------+
|    S101|Metro Mart Hyderabad| 154000|
|    S102|Metro Mart Bangalore|  65000|
|    S106|     Metro Mart Pune|  38000|
|    S107|    Metro Mart Kochi|  32000|
|    S103|   Metro Mart Mumbai|  30000|
|    S104|  Metro Mart Chennai|  24000|
|    S105|    Metro Mart Delhi|  20000|
|    S108|   Metro Mart Jaipur|  10000|
+--------+--------------------+-------+



In [101]:
spark.sql("""
SELECT s.city,
       SUM(sa.sale_amount) AS revenue
FROM sales sa
JOIN stores s ON sa.store_id = s.store_id
GROUP BY s.city
ORDER BY revenue DESC
""").show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Hyderabad| 154000|
|Bangalore|  65000|
|     Pune|  38000|
|    Kochi|  32000|
|   Mumbai|  30000|
|  Chennai|  24000|
|    Delhi|  20000|
|   Jaipur|  10000|
+---------+-------+



In [102]:
spark.sql("""
SELECT i.store_id, i.product_id, i.stock_quantity, i.reorder_level
FROM inventory i
WHERE i.stock_quantity < i.reorder_level
""").show()

+--------+----------+--------------+-------------+
|store_id|product_id|stock_quantity|reorder_level|
+--------+----------+--------------+-------------+
|    S101|      P104|             3|            5|
|    S103|      P105|             2|            5|
|    S104|      P107|             4|            5|
|    S107|      P110|             1|            3|
+--------+----------+--------------+-------------+



In [103]:
spark.sql("""
SELECT sa.*
FROM sales sa
LEFT JOIN products p
ON sa.product_id = p.product_id
WHERE p.product_id IS NULL
""").show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|     Cash|
+-------+--------+----------+----------+-------------+-----------+---------+



In [104]:
spark.sql("""
SELECT p.*
FROM products p
LEFT JOIN suppliers s
ON p.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""").show()

+----------+------------+-----------+---------+-----------+----------+
|product_id|product_name|   category|    brand|supplier_id|unit_price|
+----------+------------+-----------+---------+-----------+----------+
|      P107|       Watch|    Fashion| Fastrack|       S206|      8000|
|      P108|    Backpack|    Fashion|Wildcraft|       S206|      2500|
|      P111|  Headphones|Electronics|     Sony|       S999|      3000|
|      P112|     T-Shirt|    Fashion|     Puma|       NULL|      1500|
+----------+------------+-----------+---------+-----------+----------+



In [105]:
spark.sql("""
SELECT p.product_name,
       SUM(sa.sale_amount) AS revenue
FROM sales sa
JOIN products p
ON sa.product_id = p.product_id
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 5
""").show()

+------------+-------+
|product_name|revenue|
+------------+-------+
|      Laptop| 130000|
|      Mobile|  75000|
|Refrigerator|  38000|
|        Sofa|  32000|
|       Watch|  24000|
+------------+-------+



In [108]:
spark.sql("""
SELECT payment_m,
       SUM(sale_amount) AS revenue
FROM sales
GROUP BY payment_m
ORDER BY revenue DESC
""").show()

+---------+-------+
|payment_m|revenue|
+---------+-------+
|      UPI| 190500|
|     Card| 133000|
|     Cash|  35500|
|     NULL|  14000|
+---------+-------+



In [121]:
# PART 9: FULL REFRESH AND INCREMENTAL LOAD
gold_path = "/content/gold_sales"

sales_df.write.mode("overwrite").parquet(gold_path)

In [122]:
from pyspark.sql.functions import year, month
gold_df = sales_df.withColumn("year", year("sale_date")) \
                   .withColumn("month", month("sale_date"))
gold_df.write.mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/content/gold_sales")

In [123]:
%%writefile incremental_sales_march.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_m
SA2001,S101,P101,2026-03-01,1,65000,UPI
SA2002,S102,P102,2026-03-02,2,50000,Card
SA2003,S103,P106,2026-03-03,3,13500,Cash
SA2004,S104,P107,2026-03-04,1,8000,UPI
SA2005,S105,P108,2026-03-05,2,5000,Card

Writing incremental_sales_march.csv


In [125]:
incremental_sales_df = spark.read.csv(
    "incremental_sales_march.csv", header=True, inferSchema=True
)

incremental_sales_df.write.mode("overwrite").parquet("/content/incremental_sales_march")

In [126]:
inc_df = spark.read.parquet("/content/incremental_sales_march")
inc_df.show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA2001|    S101|      P101|2026-03-01|            1|      65000|      UPI|
| SA2002|    S102|      P102|2026-03-02|            2|      50000|     Card|
| SA2003|    S103|      P106|2026-03-03|            3|      13500|     Cash|
| SA2004|    S104|      P107|2026-03-04|            1|       8000|      UPI|
| SA2005|    S105|      P108|2026-03-05|            2|       5000|     Card|
+-------+--------+----------+----------+-------------+-----------+---------+



In [127]:
inc_clean_df = inc_df.fillna(
    {"sale_amount": 0, "payment_m": "Not Provided"}
)
inc_clean_df = inc_clean_df.withColumn(
    "data_quality_status",
    when(
        (col("sale_amount") == 0) |
        (col("payment_m") == "Not Provided"),
        "Issue"
    ).otherwise("Valid")
)
inc_clean_df.write.mode("append").parquet("silver/sales")


In [128]:
updated_sales_df = spark.read.parquet("silver/sales")
updated_sales_products_df = updated_sales_df.join(
    products_clean_df, on="product_id", how="left"
)
product_revenue_df = updated_sales_products_df.groupBy("product_name") \
    .agg(sum("sale_amount").alias("revenue"))
window_spec = Window.orderBy(col("revenue").desc())
product_revenue_df = product_revenue_df.withColumn("rank", rank().over(window_spec))
product_revenue_df.show()

+------------+-------+----+
|product_name|revenue|rank|
+------------+-------+----+
|      Laptop| 195000|   1|
|      Mobile| 125000|   2|
|Refrigerator|  38000|   3|
|        Sofa|  32000|   4|
|       Watch|  32000|   4|
|       Shoes|  31500|   6|
|    Backpack|  25000|   7|
|Office Chair|  14000|   8|
| Study Table|  12000|   9|
|        NULL|  10000|  10|
|  Television|      0|  11|
+------------+-------+----+



In [129]:
updated_sales_stores_df = updated_sales_df.join(
    stores_df, on="store_id", how="left"
)
store_revenue_df = updated_sales_stores_df.groupBy("store_name") \
    .agg(sum("sale_amount").alias("revenue"))
window_spec = Window.orderBy(col("revenue").desc())
store_revenue_df = store_revenue_df.withColumn("rank", rank().over(window_spec))
store_revenue_df.show()

+--------------------+-------+----+
|          store_name|revenue|rank|
+--------------------+-------+----+
|Metro Mart Hyderabad| 219000|   1|
|Metro Mart Bangalore| 115000|   2|
|   Metro Mart Mumbai|  43500|   3|
|     Metro Mart Pune|  38000|   4|
|    Metro Mart Kochi|  32000|   5|
|  Metro Mart Chennai|  32000|   5|
|    Metro Mart Delhi|  25000|   7|
|   Metro Mart Jaipur|  10000|   8|
+--------------------+-------+----+



In [130]:
updated_gold_df = updated_sales_df.withColumn("year", year("sale_date")) \
                                   .withColumn("month", month("sale_date"))
updated_gold_df.write.mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/content/gold_sales")

In [131]:
before_count = sales_clean_df.count()
after_count = updated_sales_df.count()

print("Sales count before incremental load:", before_count)
print("Sales count after incremental load:", after_count)
print("New records added:", after_count - before_count)

Sales count before incremental load: 15
Sales count after incremental load: 20
New records added: 5


In [132]:
# PART 10: FINAL GOLD REPORTS
from pyspark.sql.functions import count, countDistinct, sum

In [133]:
store_performance_df = updated_sales_stores_df.groupBy(
    "store_id", "store_name", "city", "state"
).agg(
    count("sale_id").alias("total_sales"),
    sum("sale_amount").alias("total_revenue")
)
store_performance_df.show()
store_performance_df.write.mode("overwrite").parquet("gold/store_performance_report")

+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|          store_name|     city|      state|total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|          1|        38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|          1|        32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|          5|       219000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|          3|        43500|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|          3|        25000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|          3|       115000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|          3|        32000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|          1|        10000|
+--------+--------------------+---------+-----------+-----------+-------------+



In [134]:
product_performance_df = updated_sales_products_df.groupBy(
    "product_id", "product_name", "category", "brand"
).agg(
    sum("quantity_sold").alias("total_quantity_sold"),
    sum("sale_amount").alias("total_revenue")
)
product_performance_df.show()
product_performance_df.write.mode("overwrite").parquet("gold/product_performance_report")

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|   category|       brand|total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|      P110|        Sofa|  Furniture|      Godrej|                  1|        32000|
|      P104|Office Chair|  Furniture| Featherlite|                  2|        14000|
|      P101|      Laptop|Electronics|      Lenovo|                  3|       195000|
|      P109|Refrigerator|Electronics|   Whirlpool|                  1|        38000|
|      P105| Study Table|  Furniture|Urban Ladder|                  1|        12000|
|      P107|       Watch|    Fashion|    Fastrack|                  4|        32000|
|      P102|      Mobile|Electronics|     Samsung|                  5|       125000|
|      P108|    Backpack|    Fashion|   Wildcraft|                 10|        25000|
|      P106|       Shoes|    Fashion|        Nike|               

In [135]:
inventory_reorder_df = inventory_products_df.select(
    "store_id",
    "product_id",
    "product_name",
    "stock_quantity",
    "reorder_level",
    "stock_status"
)

inventory_reorder_df.show()

inventory_reorder_df.write.mode("overwrite").parquet("gold/inventory_reorder_report")

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|    stock_status|
+--------+----------+------------+--------------+-------------+----------------+
|    S101|      P101|      Laptop|            10|            5|Sufficient Stock|
|    S101|      P102|      Mobile|            25|           10|Sufficient Stock|
|    S101|      P104|Office Chair|             3|            5|Reorder Required|
|    S102|      P101|      Laptop|             8|            5|Sufficient Stock|
|    S102|      P103|  Television|             5|            4|Sufficient Stock|
|    S103|      P105| Study Table|             2|            5|Reorder Required|
|    S103|      P106|       Shoes|            30|           10|Sufficient Stock|
|    S104|      P107|       Watch|             4|            5|Reorder Required|
|    S105|      P108|    Backpack|            50|           20|Sufficient Stock|
|    S106|      P109|Refrige

In [136]:
supplier_quality_df = products_suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    "supplier_quality",
    "phone",
    "email"
).distinct()
supplier_quality_df.show()
supplier_quality_df.write.mode("overwrite").parquet("gold/supplier_quality_report")

+-----------+--------------------+---------+------+----------------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|supplier_quality|       phone|               email|
+-----------+--------------------+---------+------+----------------+------------+--------------------+
|       S204|  Urban Furniture Co|    Delhi|   4.0|            Good|  9876500014|      urban@mail.com|
|       S201|    TechSource India|Hyderabad|   4.5|       Excellent|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|            Good|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|            Good|  9876500013|        Not Provided|
|       S205|      Fashion Direct|     Pune|   3.8|         Average|Not Provided|        Not Provided|
|       S999|                NULL|     NULL|  NULL|         Average|        NULL|                NULL|
|       S206|                NULL|     NULL|  NULL|         Average|     

In [137]:
category_revenue_df = updated_sales_products_df.groupBy("category").agg(
    countDistinct("product_id").alias("total_products"),
    sum("quantity_sold").alias("total_quantity_sold"),
    sum("sale_amount").alias("total_revenue")
)
category_revenue_df.show()
category_revenue_df.write.mode("overwrite").parquet("gold/category_revenue_report")

+-----------+--------------+-------------------+-------------+
|   category|total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|    Fashion|             3|                 21|        88500|
|       NULL|             1|                  2|        10000|
|Electronics|             4|                 10|       358000|
|  Furniture|             3|                  4|        58000|
+-----------+--------------+-------------------+-------------+



In [138]:
payment_mode_df = updated_sales_df.groupBy("payment_m").agg(
    count("sale_id").alias("total_transactions"),
    sum("sale_amount").alias("total_revenue")
).withColumnRenamed("payment_m", "payment_mode")
payment_mode_df.show()
payment_mode_df.write.mode("overwrite").parquet("gold/payment_mode_report")

+------------+------------------+-------------+
|payment_mode|total_transactions|total_revenue|
+------------+------------------+-------------+
|        Card|                 7|       188000|
|        Cash|                 4|        49000|
|Not Provided|                 1|        14000|
|         UPI|                 8|       263500|
+------------+------------------+-------------+

